# Hands-On 6: Comparing classification performance

Record a prediction before running each experiment.

In [ ]:
from pathlib import Path
import os
import sys
try:
    import mlcourse.setup
except ModuleNotFoundError:
    bases = [Path(os.environ.get("MLCOURSE_ROOT", Path.cwd())), Path.cwd(), Path("/content/pp-machine-learning")]
    for base in bases:
        for candidate in (base.resolve(), *base.resolve().parents):
            if (candidate / "src/mlcourse/setup.py").is_file():
                sys.path.insert(0, str(candidate / "src"))
                break
        else:
            continue
        break
    else:
        raise RuntimeError("Course files not found. Open the extracted course repository or set MLCOURSE_ROOT to its location.") from None

In [ ]:
from mlcourse.setup import setup_notebook
REPO_ROOT = setup_notebook()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from mlcourse.labs import load_course_data

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import friedmanchisquare
from mlcourse.labs import load_task_results

## 1. Inspect three tasks

Load `ds1`, `ds2` and `ds3`. Inspect their dimensions and class counts.

**Prediction:** Should one classifier necessarily perform best on all three tasks?

*Your response.*

In [ ]:
datasets = {name: load_course_data(name) for name in ['ds1', 'ds2', 'ds3']}
display(pd.DataFrame({name: frame.Class.value_counts().sort_index() for name, frame in datasets.items()}))
display(pd.DataFrame([{'dataset': name, 'rows': len(frame), 'predictors': 2} for name, frame in datasets.items()]))

**Observation:** Do these tasks have the same sizes and class representation?

*Your response.*

**Explanation:** Why can comparable class counts conceal different learning difficulty?

*Your response.*

## 2. Run shared cross-validation folds

Use ten stratified folds to evaluate seven classifiers. Fit scaling inside each training fold.

**Prediction:** Why use the same folds for every classifier?

*Your response.*

In [ ]:
models = {'k-NN': make_pipeline(StandardScaler(), KNeighborsClassifier()),
          'Tree': DecisionTreeClassifier(random_state=0),
          'LDA': LinearDiscriminantAnalysis(), 'Naive Bayes': GaussianNB(),
          **{f'SVM {kernel}': make_pipeline(StandardScaler(), SVC(kernel=kernel)) for kernel in ['linear', 'poly', 'rbf']}}
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
fold_results = {}
for name, frame in datasets.items():
    fold_results[name] = pd.DataFrame({label: cross_val_score(model, frame[['X1', 'X2']], frame.Class,
        cv=cv, scoring='accuracy') for label, model in models.items()})
display(fold_results['ds2'])

**Observation:** What does one row of this result table represent?

*Your response.*

**Explanation:** Why is scaling learned separately within each training fold?

*Your response.*

## 3. Summarise performance and variability

Compute mean accuracy and standard deviation for `ds2`.

**Prediction:** Can two models have similar means but different variability?

*Your response.*

In [ ]:
summary = pd.DataFrame({'mean_accuracy': fold_results['ds2'].mean(),
                        'standard_deviation': fold_results['ds2'].std(ddof=1)})
display(summary.sort_values('mean_accuracy', ascending=False))

**Observation:** Find a close mean comparison and compare its standard deviations.

*Your response.*

**Explanation:** What information is lost when only the mean is reported?

*Your response.*

## 4. Compare the score distributions

Plot the fold-wise results. Use the scores to describe estimated performance and its variability.

**Prediction:** Will the ordering by mean describe every fold?

*Your response.*

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), layout='constrained')
fold_results['ds2'].plot.box(ax=ax, rot=25)
ax.set(xlabel='Classifier', ylabel='Fold accuracy', title='Ten held-out folds on ds2')
plt.show()
display(pd.DataFrame({name: scores.mean() for name, scores in fold_results.items()}))

**Observation:** Compare distribution overlap and the ordering across the three tasks.

*Your response.*

**Explanation:** Why should a small mean difference be interpreted with the observed variability?

*Your response.*

## 5. Compare ten dataset-level results

Load the provided results for ten tasks. Average within each task and rank classifiers, with rank `1` assigned to the highest accuracy.

**Prediction:** Will averaging accuracy and averaging ranks necessarily give the same ordering?

*Your response.*

In [ ]:
task_means = load_task_results()
ranks = task_means.rank(axis=1, ascending=False, method='average')
display(task_means)
display(ranks)

**Observation:** Identify a tied rank and a change in ordering between tasks.

*Your response.*

**Explanation:** What changes when one observation is an entire dataset?

*Your response.*

## 6. Summarise ranks and the Friedman test

Run one Friedman test on the dataset-level means and plot average ranks.

**Prediction:** Does the algorithm with best average rank necessarily win every task?

*Your response.*

In [ ]:
statistic, pvalue = friedmanchisquare(*(task_means[column] for column in task_means))
print(f'Friedman statistic: {statistic:.3f}; p-value: {pvalue:.3g}')
mean_ranks = ranks.mean().sort_values()
fig, ax = plt.subplots(figsize=(9, 5), layout='constrained')
mean_ranks.plot.barh(ax=ax, color='#0072B2')
ax.invert_yaxis()
ax.set(xlabel='Average rank (lower is better)', ylabel='Classifier', title='Comparison across ten tasks')
plt.show()
display(mean_ranks.rename('average_rank'))

**Observation:** Report the best average rank and interpret the p-value at `0.050`.

*Your response.*

**Explanation:** What does an omnibus result establish about these algorithms?

*Your response.*